In [3]:
from transformers import AutoTokenizer, AutoModel, AutoConfig, PreTrainedModel, Trainer, TrainingArguments
import torch, torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from src.preprocess import QuestPreprocessor
from src.postprocess import QuestPostprocessor
from src.loss import SpearmanLoss
import os

MODEL_NAME = "answerdotai/ModernBERT-base"
TARGET_COLS = ['question_asker_intent_understanding',
 'question_body_critical',
 'question_conversational',
 'question_expect_short_answer',
 'question_fact_seeking',
 'question_has_commonly_accepted_answer',
 'question_interestingness_others',
 'question_interestingness_self',
 'question_multi_intent',
 'question_not_really_a_question',
 'question_opinion_seeking',
 'question_type_choice',
 'question_type_compare',
 'question_type_consequence',
 'question_type_definition',
 'question_type_entity',
 'question_type_instructions',
 'question_type_procedure',
 'question_type_reason_explanation',
 'question_type_spelling',
 'question_well_written',
 'answer_helpful',
 'answer_level_of_information',
 'answer_plausible',
 'answer_relevance',
 'answer_satisfaction',
 'answer_type_instructions',
 'answer_type_procedure',
 'answer_type_reason_explanation',
 'answer_well_written']

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class QuestDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        # We assume 'text_input' column is already created by the preprocessor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1) Get the pre-processed text
        # Ensure it's a string in case of NaNs
        text = str(row['text_input'])

        # 2) Tokenize
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze() for k, v in enc.items()}

        # 3) Labels (Only if they exist in the dataframe)
        # We check if the first target column exists
        if TARGET_COLS[0] in row:
            labels = torch.tensor(row[TARGET_COLS].astype(float).values, dtype=torch.float32)
            item["labels"] = labels

        return item

class CustomModel(PreTrainedModel):
    def __init__(self, config, n_targets=30, dropout_prob=0.1, freeze_encoder=False):
        super().__init__(config)
        if config._name_or_path:
            self.encoder = AutoModel.from_pretrained(config._name_or_path)
        else:
            self.encoder = AutoModel.from_config(config)
        # freeze encoder if needed
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        self.dropout = nn.Dropout(dropout_prob)
        self.regressor = nn.Linear(config.hidden_size, n_targets)
        self.sigmoid = nn.Sigmoid()
        self.spearman_loss = SpearmanLoss()
        # init weights
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, token_type_ids=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # some models have pooler_output, some don't
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            pooled_output = outputs.pooler_output
        else:
            pooled_output = outputs.last_hidden_state[:, 0, :]

        x = self.dropout(pooled_output)

        raw_logits = self.regressor(x)

        preds = self.sigmoid(raw_logits)
        loss = None
        if labels is not None:
            #loss_mse = nn.MSELoss()(preds, labels)
            loss_bce = nn.BCEWithLogitsLoss()(raw_logits, labels)
            loss_spearman = self.spearman_loss(preds, labels)
            #loss = loss_mse
            loss = loss_bce * 0.9 + loss_spearman * 0.1

        return {"loss": loss, "logits": preds}

config = AutoConfig.from_pretrained(MODEL_NAME)
model = CustomModel(config, n_targets=len(TARGET_COLS))

In [4]:
# --- TRAINING BLOCK ---
import numpy as np
from scipy.stats import spearmanr

# 1. Load Train Data
print('Loading train.csv...')
raw_train_df = pd.read_csv('train.csv')

# --- NEW PREPROCESSING STEP ---
preprocessor = QuestPreprocessor(use_category=True, use_host=True)
train_df_processed = preprocessor.preprocess(raw_train_df)
# ------------------------------

postprocessor = QuestPostprocessor(train_labels=raw_train_df[TARGET_COLS].values)


train_split, val_split = train_test_split(train_df_processed, test_size=0.1, random_state=42)

train_ds = QuestDataset(train_split, tokenizer)
val_ds = QuestDataset(val_split, tokenizer)

# 2. Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    logits = postprocessor.optimize_ranks(logits)
    
    rhos = []
    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(logits[:, i]) == 0:
            rhos.append(0.0)
        else:
            rho, _ = spearmanr(labels[:, i], logits[:, i])
            rhos.append(rho)
    return {'spearman': np.nanmean(rhos)}

# 3. Train
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
	fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Starting Training...')
trainer.train()
print('Training Complete. Evaluating...')
eval_res = trainer.evaluate()
print(f'Validation Spearman Rho: {eval_res["eval_spearman"]:.4f}')

# 4. SAVE MODEL FOR INFERENCE
output_model_dir = './saved_model'
if not os.path.exists(output_model_dir):
    os.makedirs(output_model_dir)

# Save the model weights
trainer.save_model(output_model_dir)
# Save the tokenizer
tokenizer.save_pretrained(output_model_dir)
# Save the config
config.save_pretrained(output_model_dir)
print(f"Model, tokenizer, and config saved to {output_model_dir}")

Loading train.csv...
Starting Training...


Epoch,Training Loss,Validation Loss,Spearman
1,0.410500,0.397447,0.380994
2,0.384900,0.388720,0.408510
3,0.368300,0.390089,0.415030
4,0.348700,0.391659,0.416154
5,0.319100,0.400144,0.403424
6,0.296600,0.408918,0.390185
7,0.273200,0.415100,0.398383
8,0.262600,0.422127,0.385341
9,0.255100,0.428115,0.389207
10,0.248100,0.433218,0.383068


Training Complete. Evaluating...


Validation Spearman Rho: 0.4085
Model, tokenizer, and config saved to ./saved_model
